# Uniform-baseline Budget/Runtime Trade-off

This notebook demonstrates how the budget/runtime trade-off analysis presented in the paper is generated. It reproduces the figures and tables related to materialized-view selection under different budgets, including:

1. **Best workload saving per budget for every strategy family**, comparing single-view and multi-view rewrites.
2. **Query saving versus total refresh cost per budget**, comparing schema-driven and query-driven materialized-view selection.

The notebook loads the experimental results produced by the benchmark scripts, performs the budget optimization analysis, and generates the figures and summary tables reported in the paper.

## 1. Best workload saving per budget for every strategy family

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import csv
import json
import os
import tempfile

import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp


def locate_analysis_dir(start: Path | None = None) -> Path:
    """Locate tpch_sf_01/analysis from common notebook launch directories."""
    start = (start or Path.cwd()).resolve()
    markers = (
        Path("data/query_driven/all_results_mv_05.json"),
        Path("data/schema_driven/all_results_multi_view_05.json"),
    )
    candidates: list[Path] = []
    for anchor in (start, *start.parents):
        candidates.extend(
            (anchor, anchor / "analysis", anchor / "tpch_sf_01" / "analysis")
        )

    for candidate in dict.fromkeys(path.resolve() for path in candidates):
        if all((candidate / marker).is_file() for marker in markers):
            return candidate

    raise FileNotFoundError(
        "Cannot locate tpch_sf_01/analysis from the current working directory."
    )


ANALYSIS_DIR = locate_analysis_dir()
DATA_DIR = ANALYSIS_DIR / "data"
RESULT_DIR = ANALYSIS_DIR / "result"
TRADE_OFF_OUTPUT_DIR = RESULT_DIR / "trade_off_template_time_uniform"
TRADE_OFF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PLOT = TRADE_OFF_OUTPUT_DIR / "all_budget_tradeoff_saved_time.png"
OUTPUT_PLOT_PDF = TRADE_OFF_OUTPUT_DIR / "all_budget_tradeoff_saved_time.pdf"
OUTPUT_SUMMARY = TRADE_OFF_OUTPUT_DIR / "all_budget_tradeoff_summary.md"
OUTPUT_BEST_CSV = TRADE_OFF_OUTPUT_DIR / "all_budget_tradeoff_best_by_budget.csv"
OUTPUT_BASELINE_CSV = TRADE_OFF_OUTPUT_DIR / "unified_baseline_by_query.csv"

CACHE_DIR = Path(tempfile.gettempdir()) / "tpch_join_experiment_matplotlib_cache"
MPL_CONFIG_DIR = CACHE_DIR / "matplotlib"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

MAX_VIEW_COUNT = 7
WORKLOAD_QIDS = [qid for qid in range(1, 23) if qid not in {1, 6}]
GLOBAL_SINGLE_MV_RELEVANT_QIDS = (2, 5, 7, 8, 9, 13, 18, 20)
PLOT_FIG_SIZE = (12.5, 3.2)

AXIS_LABEL_FONT_SIZE = 15
TICK_LABEL_FONT_SIZE = 10
LEGEND_FONT_SIZE = 13
LEGEND_ANCHOR = (1.02, 0.5)
PLOT_RIGHT_MARGIN = 0.78

BASELINE_SOURCE_PATHS = (
    DATA_DIR / "query_driven" / "all_results_mv_05.json",
    DATA_DIR / "query_driven" / "all_results_multi_05.json",
    DATA_DIR / "query_driven" / "all_results_view.json",
    DATA_DIR / "query_driven" / "all_results_multi_view_05.json",
    DATA_DIR / "schema_driven" / "all_results_mv_05.json",
    DATA_DIR / "schema_driven" / "all_results_multi_mv_05.json",
    DATA_DIR / "schema_driven" / "all_results_view_05.json",
    DATA_DIR / "schema_driven" / "all_results_multi_view_05.json",
)
_UNIFIED_BASELINE_CACHE: dict[int, float] | None = None


def relative_to_analysis(path: Path) -> str:
    return path.resolve().relative_to(ANALYSIS_DIR).as_posix()


def validate_input_paths(paths) -> None:
    missing = [relative_to_analysis(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing input files:\n  - " + "\n  - ".join(missing))


print("Analysis directory located.")
print(f"Input directory: {relative_to_analysis(DATA_DIR)}")
print(f"Result root: {relative_to_analysis(RESULT_DIR)}")
print(
    "Trade-off output directory: "
    f"{relative_to_analysis(TRADE_OFF_OUTPUT_DIR)}"
)


In [ ]:
@dataclass(frozen=True)
class DatasetSpec:
    slug: str
    label: str
    path: Path
    metric_key: str
    color: str
    marker: str
    linestyle: str
    additional_paths: tuple[Path, ...] = field(default_factory=tuple)
    optimization_qids: tuple[int, ...] | None = None

    @property
    def paths(self) -> tuple[Path, ...]:
        return self.additional_paths + (self.path,)

    @property
    def active_qids(self) -> tuple[int, ...]:
        return self.optimization_qids or tuple(WORKLOAD_QIDS)


@dataclass
class DatasetAnalysis:
    spec: DatasetSpec
    baseline_by_query: dict[int, float]
    baseline_total: float
    view_profiles: dict[str, dict[str, Any]]
    dominated_details: dict[str, dict[str, Any]]
    best_by_view_count: dict[int, dict[str, Any]]
    enumerated: int
    raw_candidate_count: int

    @property
    def query_count(self) -> int:
        return len(self.baseline_by_query)

    @property
    def retained_candidate_count(self) -> int:
        return len(self.view_profiles)

    @property
    def pruned_candidate_count(self) -> int:
        return len(self.dominated_details)


DATASETS = [
    DatasetSpec(
        slug="query_driven_single_mv",
        label="QD single MV",
        path=DATA_DIR / "query_driven" / "all_results_mv_05.json",
        metric_key="materialized_view",
        color="#1F77B4",
        marker="o",
        linestyle="-",
        optimization_qids=GLOBAL_SINGLE_MV_RELEVANT_QIDS,
    ),
    DatasetSpec(
        slug="query_driven_multi_mv",
        label="QD multi MV",
        path=DATA_DIR / "query_driven" / "all_results_multi_05.json",
        metric_key="materialized_view",
        color="#1F77B4",
        marker="s",
        linestyle="--",
        additional_paths=(DATA_DIR / "query_driven" / "all_results_mv_05.json",),
    ),
    DatasetSpec(
        slug="query_driven_single_view",
        label="QD single view",
        path=DATA_DIR / "query_driven" / "all_results_view.json",
        metric_key="view",
        color="#2CA02C",
        marker="^",
        linestyle="-",
    ),
    DatasetSpec(
        slug="query_driven_multi_view",
        label="QD multi view",
        path=DATA_DIR / "query_driven" / "all_results_multi_view_05.json",
        metric_key="view",
        color="#2CA02C",
        marker="D",
        linestyle="--",
        additional_paths=(DATA_DIR / "query_driven" / "all_results_view.json",),
    ),
    DatasetSpec(
        slug="schema_driven_single_mv",
        label="SD single MV",
        path=DATA_DIR / "schema_driven" / "all_results_mv_05.json",
        metric_key="materialized_view",
        color="#FF7F0E",
        marker="o",
        linestyle="-",
        optimization_qids=GLOBAL_SINGLE_MV_RELEVANT_QIDS,
    ),
    DatasetSpec(
        slug="schema_driven_multi_mv",
        label="SD multi MV",
        path=DATA_DIR / "schema_driven" / "all_results_multi_mv_05.json",
        metric_key="materialized_view",
        color="#FF7F0E",
        marker="s",
        linestyle="--",
        additional_paths=(DATA_DIR / "schema_driven" / "all_results_mv_05.json",),
    ),
    DatasetSpec(
        slug="schema_driven_single_view",
        label="SD single view",
        path=DATA_DIR / "schema_driven" / "all_results_view_05.json",
        metric_key="view",
        color="#D62728",
        marker="^",
        linestyle="-",
    ),
    DatasetSpec(
        slug="schema_driven_multi_view",
        label="SD multi view",
        path=DATA_DIR / "schema_driven" / "all_results_multi_view_05.json",
        metric_key="view",
        color="#D62728",
        marker="D",
        linestyle="--",
        additional_paths=(DATA_DIR / "schema_driven" / "all_results_view_05.json",),
    ),
]


In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_dataset_sources(spec: DatasetSpec) -> list[tuple[Path, dict[str, Any]]]:
    return [(path, load_json(path)) for path in spec.paths]


def source_paths_label(spec: DatasetSpec) -> str:
    return " + ".join(relative_to_analysis(path) for path in spec.paths)


def load_unified_baseline() -> dict[int, float]:
    """Average within each file, then across files where each query appears."""
    global _UNIFIED_BASELINE_CACHE
    if _UNIFIED_BASELINE_CACHE is not None:
        return _UNIFIED_BASELINE_CACHE.copy()

    validate_input_paths(BASELINE_SOURCE_PATHS)
    per_query_file_means: dict[int, list[float]] = defaultdict(list)

    for path in BASELINE_SOURCE_PATHS:
        values_by_query: dict[int, list[float]] = defaultdict(list)
        for job_key, block in load_json(path).items():
            if not isinstance(block, dict):
                continue
            qid, _ = parse_key(job_key)
            baseline = block.get("baseline")
            if qid in WORKLOAD_QIDS and isinstance(baseline, dict):
                values_by_query[qid].append(float(baseline["average_time"]))

        for qid, values in values_by_query.items():
            if values:
                per_query_file_means[qid].append(float(np.mean(values)))

    incomplete_qids = [
        qid
        for qid in WORKLOAD_QIDS
        if not per_query_file_means[qid]
    ]
    if incomplete_qids:
        formatted = ", ".join(f"Q{qid}" for qid in incomplete_qids)
        raise RuntimeError(f"Cannot build the unified baseline for: {formatted}")

    _UNIFIED_BASELINE_CACHE = {
        qid: float(np.mean(per_query_file_means[qid]))
        for qid in WORKLOAD_QIDS
    }
    return _UNIFIED_BASELINE_CACHE.copy()


In [ ]:
def parse_key(job_key: str) -> tuple[int, str]:
    prefix, tag = job_key.split("_", 1)
    return int(prefix[1:]), tag


def canonicalize_single_view_tag(view_tag: str) -> str:
    parts = view_tag.split("_")
    if "N1" in parts and "N2" in parts:
        return view_tag
    normalized_parts = ["N" if part in {"N1", "N2"} else part for part in parts]
    return "_".join(normalized_parts)


def canonicalize_view_tag(view_tag: str) -> str:
    return "__".join(canonicalize_single_view_tag(part) for part in view_tag.split("__"))


def required_view_tags(view_tag: str) -> tuple[str, ...]:
    return tuple(part for part in canonicalize_view_tag(view_tag).split("__") if part)


def format_seconds(value: float) -> str:
    return f"{value:.6f}s"


def format_percent(value: float) -> str:
    return f"{value * 100:.2f}%"


def metric_average_time(block: dict[str, Any], metric_key: str, job_key: str) -> float:
    metric = block.get(metric_key)
    if not isinstance(metric, dict):
        available = ", ".join(sorted(block))
        raise KeyError(
            f"{job_key} does not contain metric '{metric_key}'. "
            f"Available top-level keys: {available}"
        )

    return float(metric["average_time"])


def collect_query_data(
    data_sources: list[tuple[Path, dict[str, Any]]],
    metric_key: str,
) -> tuple[dict[int, float], dict[int, list[dict[str, Any]]], float, int]:
    baseline_by_query = load_unified_baseline()
    raw_entries: dict[int, list[tuple[str, str, float]]] = defaultdict(list)

    for _, data in data_sources:
        for job_key, block in data.items():
            if not isinstance(block, dict):
                continue

            qid, tag = parse_key(job_key)
            if qid not in baseline_by_query:
                continue

            canonical_tag = canonicalize_view_tag(tag)
            candidate_time = metric_average_time(block, metric_key, job_key)
            raw_entries[qid].append((canonical_tag, tag, candidate_time))

    improving_by_query: dict[int, list[dict[str, Any]]] = {}
    raw_candidate_tags: set[str] = set()
    for qid in WORKLOAD_QIDS:
        canonical_baseline = baseline_by_query[qid]
        best_by_tag: dict[str, float] = {}
        aliases_by_tag: dict[str, set[str]] = defaultdict(set)

        for canonical_tag, original_tag, candidate_time in raw_entries[qid]:
            raw_candidate_tags.add(canonical_tag)
            aliases_by_tag[canonical_tag].add(original_tag)
            current = best_by_tag.get(canonical_tag)
            if current is None or candidate_time < current:
                best_by_tag[canonical_tag] = candidate_time

        improving_entries = []
        for canonical_tag, candidate_time in best_by_tag.items():
            if candidate_time >= canonical_baseline:
                continue
            improving_entries.append(
                {
                    "tag": canonical_tag,
                    "candidate_time": candidate_time,
                    "baseline_time": canonical_baseline,
                    "gain_ratio": (canonical_baseline - candidate_time) / canonical_baseline,
                    "saved_time": canonical_baseline - candidate_time,
                    "aliases": sorted(aliases_by_tag[canonical_tag]),
                    "required_views": required_view_tags(canonical_tag),
                }
            )

        improving_entries.sort(key=lambda item: (item["candidate_time"], item["tag"]))
        improving_by_query[qid] = improving_entries

    baseline_total = sum(baseline_by_query[qid] for qid in WORKLOAD_QIDS)
    return baseline_by_query, improving_by_query, baseline_total, len(raw_candidate_tags)


def build_view_profiles(
    baseline_by_query: dict[int, float],
    improving_by_query: dict[int, list[dict[str, Any]]],
) -> tuple[dict[str, dict[str, Any]], dict[str, dict[str, Any]]]:
    profiles: dict[str, dict[str, Any]] = {}

    for qid, entries in improving_by_query.items():
        for item in entries:
            tag = item["tag"]
            profile = profiles.setdefault(
                tag,
                {
                    "times": {query: float("inf") for query in baseline_by_query},
                    "queries": set(),
                    "aliases": set(),
                    "required_views": required_view_tags(tag),
                },
            )
            profile["times"][qid] = min(profile["times"][qid], item["candidate_time"])
            profile["queries"].add(qid)
            profile["aliases"].update(item["aliases"])

    query_ids = sorted(baseline_by_query)

    for tag, profile in profiles.items():
        profile["queries"] = sorted(profile["queries"])
        profile["aliases"] = sorted(profile["aliases"])
        profile["improvements"] = {
            qid: baseline_by_query[qid] - profile["times"][qid]
            for qid in query_ids
            if profile["times"][qid] < baseline_by_query[qid]
        }

    return profiles, {}


In [ ]:
def baseline_option(qid: int, baseline_by_query: dict[int, float]) -> dict[str, Any]:
    return {
        "tag": "baseline",
        "candidate_time": baseline_by_query[qid],
        "required_views": tuple(),
    }


def solve_best_for_budget(
    improving_by_query: dict[int, list[dict[str, Any]]],
    baseline_by_query: dict[int, float],
    baseline_total: float,
    base_view_candidates: list[str],
    budget: int,
    active_qids: tuple[int, ...],
) -> dict[str, Any]:
    query_ids = sorted(active_qids)
    view_index = {view_name: idx for idx, view_name in enumerate(base_view_candidates)}

    option_groups: list[tuple[int, list[dict[str, Any]]]] = []
    option_offsets: list[int] = []
    option_count = 0
    for qid in query_ids:
        options = [baseline_option(qid, baseline_by_query)] + improving_by_query.get(qid, [])
        option_offsets.append(option_count)
        option_count += len(options)
        option_groups.append((qid, options))

    num_x = len(base_view_candidates)
    num_y = option_count
    num_vars = num_x + num_y

    objective = np.zeros(num_vars, dtype=float)
    objective[:num_x] = 1e-9
    constraints = []
    rhs_lower = []
    rhs_upper = []

    for group_idx, (_, options) in enumerate(option_groups):
        row = np.zeros(num_vars, dtype=float)
        base_offset = num_x + option_offsets[group_idx]
        for option_idx, option in enumerate(options):
            row[base_offset + option_idx] = 1.0
            objective[base_offset + option_idx] = option["candidate_time"]
        constraints.append(row)
        rhs_lower.append(1.0)
        rhs_upper.append(1.0)

    budget_row = np.zeros(num_vars, dtype=float)
    budget_row[:num_x] = 1.0
    constraints.append(budget_row)
    rhs_lower.append(0.0)
    rhs_upper.append(float(budget))

    for group_idx, (_, options) in enumerate(option_groups):
        base_offset = num_x + option_offsets[group_idx]
        for option_idx, option in enumerate(options):
            y_idx = base_offset + option_idx
            for view_name in option["required_views"]:
                row = np.zeros(num_vars, dtype=float)
                row[y_idx] = 1.0
                row[view_index[view_name]] = -1.0
                constraints.append(row)
                rhs_lower.append(-np.inf)
                rhs_upper.append(0.0)

    result = milp(
        c=objective,
        integrality=np.ones(num_vars, dtype=int),
        bounds=Bounds(np.zeros(num_vars), np.ones(num_vars)),
        constraints=LinearConstraint(np.vstack(constraints), np.array(rhs_lower), np.array(rhs_upper)),
    )
    if not result.success or result.x is None:
        raise RuntimeError(f"MILP failed for budget {budget}: {result.message}")

    solution = result.x
    best_view_by_query: dict[int, str] = {}
    best_runtime_by_query: dict[int, float] = {}
    required_views: set[str] = set()
    used_templates: set[str] = set()

    for group_idx, (qid, options) in enumerate(option_groups):
        base_offset = num_x + option_offsets[group_idx]
        chosen_option = None
        for option_idx, option in enumerate(options):
            if solution[base_offset + option_idx] >= 0.5:
                chosen_option = option
                break
        if chosen_option is None:
            raise RuntimeError(f"MILP did not choose an option for Q{qid} at budget {budget}.")

        best_view_by_query[qid] = chosen_option["tag"]
        best_runtime_by_query[qid] = chosen_option["candidate_time"]
        if chosen_option["tag"] != "baseline":
            used_templates.add(chosen_option["tag"])
            required_views.update(chosen_option["required_views"])

    view_set = tuple(sorted(required_views))
    total_runtime = baseline_total - sum(baseline_by_query[qid] for qid in query_ids)
    total_runtime += sum(best_runtime_by_query.values())
    covered_queries = sorted(qid for qid in query_ids if best_view_by_query[qid] != "baseline")
    return {
        "budget": budget,
        "view_count": len(view_set),
        "view_set": view_set,
        "used_templates": tuple(sorted(used_templates)),
        "covered_queries": covered_queries,
        "covered_query_count": len(covered_queries),
        "best_view_by_query": best_view_by_query,
        "best_runtime_by_query": best_runtime_by_query,
        "total_runtime": total_runtime,
        "saved_time": baseline_total - total_runtime,
        "gain_ratio": (baseline_total - total_runtime) / baseline_total if baseline_total else 0.0,
    }


def enumerate_view_sets(
    improving_by_query: dict[int, list[dict[str, Any]]],
    baseline_by_query: dict[int, float],
    baseline_total: float,
    active_qids: tuple[int, ...],
) -> tuple[dict[int, dict[str, Any]], int]:
    base_view_candidates = sorted(
        {
            view_name
            for qid, entries in improving_by_query.items()
            if qid in active_qids
            for item in entries
            for view_name in item["required_views"]
        }
    )

    best_by_view_count = {}
    for budget in range(0, MAX_VIEW_COUNT + 1):
        best_by_view_count[budget] = solve_best_for_budget(
            improving_by_query,
            baseline_by_query,
            baseline_total,
            base_view_candidates,
            budget,
            active_qids,
        )

    return best_by_view_count, len(best_by_view_count)


def analyze_dataset(spec: DatasetSpec) -> DatasetAnalysis:
    data_sources = load_dataset_sources(spec)
    baseline_by_query, improving_by_query, baseline_total, raw_candidate_count = collect_query_data(
        data_sources,
        spec.metric_key,
    )
    view_profiles, dominated_details = build_view_profiles(baseline_by_query, improving_by_query)
    best_by_view_count, enumerated = enumerate_view_sets(
        improving_by_query,
        baseline_by_query,
        baseline_total,
        spec.active_qids,
    )

    return DatasetAnalysis(
        spec=spec,
        baseline_by_query=baseline_by_query,
        baseline_total=baseline_total,
        view_profiles=view_profiles,
        dominated_details=dominated_details,
        best_by_view_count=best_by_view_count,
        enumerated=enumerated,
        raw_candidate_count=raw_candidate_count,
    )


In [ ]:
def view_set_label(view_set: tuple[str, ...]) -> str:
    if not view_set:
        return "baseline"
    return " | ".join(view_set)


def write_best_csv(analyses: list[DatasetAnalysis]) -> None:
    fieldnames = [
        "dataset",
        "label",
        "source_path",
        "metric_key",
        "budget",
        "workload_queries",
        "covered_queries",
        "created_view_count",
        "raw_candidates",
        "retained_candidates",
        "pruned_candidates",
        "optimization_runs",
        "baseline_runtime",
        "total_runtime",
        "saved_time",
        "gain_ratio",
        "view_set",
        "used_templates",
    ]

    with OUTPUT_BEST_CSV.open("w", encoding="utf-8", newline="") as csv_out:
        writer = csv.DictWriter(csv_out, fieldnames=fieldnames)
        writer.writeheader()

        for analysis in analyses:
            for budget in range(0, MAX_VIEW_COUNT + 1):
                best = analysis.best_by_view_count.get(budget)
                if best is None:
                    continue

                writer.writerow(
                    {
                        "dataset": analysis.spec.slug,
                        "label": analysis.spec.label,
                        "source_path": source_paths_label(analysis.spec),
                        "metric_key": analysis.spec.metric_key,
                        "budget": budget,
                        "workload_queries": analysis.query_count,
                        "covered_queries": best["covered_query_count"],
                        "created_view_count": best["view_count"],
                        "raw_candidates": analysis.raw_candidate_count,
                        "retained_candidates": analysis.retained_candidate_count,
                        "pruned_candidates": analysis.pruned_candidate_count,
                        "optimization_runs": analysis.enumerated,
                        "baseline_runtime": round(analysis.baseline_total, 12),
                        "total_runtime": round(best["total_runtime"], 12),
                        "saved_time": round(best["saved_time"], 12),
                        "gain_ratio": round(best["gain_ratio"], 12),
                        "view_set": view_set_label(best["view_set"]),
                        "used_templates": view_set_label(best["used_templates"]),
                    }
                )


def build_summary(analyses: list[DatasetAnalysis]) -> str:
    lines = [
        "# Budget Runtime Tradeoff Summary",
        "",
        (
            "Unified baseline: rebuilt from the eight local JSON inputs using "
            "mean-of-per-file-query-means and saved to "
            f"{relative_to_analysis(OUTPUT_BASELINE_CSV)}."
        ),
        f"Workload queries: {', '.join(f'Q{qid}' for qid in WORKLOAD_QIDS)}.",
        "Candidate runtimes are loaded from each source JSON, while baseline runtimes always come from the unified workload baseline.",
        "Single `N1` or `N2` aliases are canonicalized to `N`; candidate tags containing both `N1` and `N2` are kept distinct. Multi-tag candidates joined by `__` apply this rule per component.",
        "Dashed multi-series combine the corresponding single-candidate and multi-candidate files, and the budget is the number of distinct base MVs/views created.",
        f"Single-MV series use the same relevant-query optimization subset as the global comparison source: {', '.join(f'Q{qid}' for qid in GLOBAL_SINGLE_MV_RELEVANT_QIDS)}.",
        "",
        f"| Dataset | Source | Metric | Workload queries | Raw candidates | Retained | Pruned | Baseline | Best @ budget {MAX_VIEW_COUNT} | Gain @ budget {MAX_VIEW_COUNT} |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|",
    ]

    for analysis in analyses:
        max_budget = max(analysis.best_by_view_count)
        best = analysis.best_by_view_count[max_budget]
        lines.append(
            f"| {analysis.spec.label} | "
            f"`{source_paths_label(analysis.spec)}` | "
            f"`{analysis.spec.metric_key}` | "
            f"{analysis.query_count} | "
            f"{analysis.raw_candidate_count} | "
            f"{analysis.retained_candidate_count} | "
            f"{analysis.pruned_candidate_count} | "
            f"{format_seconds(analysis.baseline_total)} | "
            f"{format_seconds(best['total_runtime'])} | "
            f"{format_percent(best['gain_ratio'])} |"
        )

    lines.extend(
        [
            "",
            "## Best Runtime By Budget",
            "",
            "| Dataset | Budget | Created views | Covered queries | Best total runtime | Time saved | Gain | Created MV/View set |",
            "|---|---:|---:|---:|---:|---:|---:|---|",
        ]
    )

    for analysis in analyses:
        for budget in range(0, MAX_VIEW_COUNT + 1):
            best = analysis.best_by_view_count.get(budget)
            if best is None:
                continue

            view_label = "`baseline`" if budget == 0 else ", ".join(
                f"`{tag}`" for tag in best["view_set"]
            )
            lines.append(
                f"| {analysis.spec.label} | "
                f"{budget} | "
                f"{best['view_count']} | "
                f"{best['covered_query_count']} | "
                f"{format_seconds(best['total_runtime'])} | "
                f"{format_seconds(best['saved_time'])} | "
                f"{format_percent(best['gain_ratio'])} | "
                f"{view_label} |"
            )

    return "\n".join(lines) + "\n"


def plot_combined_tradeoff(analyses: list[DatasetAnalysis]) -> None:
    fig, ax = plt.subplots(figsize=PLOT_FIG_SIZE)
    legend_handles = []
    legend_labels = []

    for analysis in analyses:
        budgets = []
        saved_times = []
        for budget in range(0, MAX_VIEW_COUNT + 1):
            best = analysis.best_by_view_count.get(budget)
            if best is None:
                continue
            budgets.append(budget)
            saved_times.append(best["saved_time"])

        (line,) = ax.plot(
            budgets,
            saved_times,
            color=analysis.spec.color,
            linestyle=analysis.spec.linestyle,
            linewidth=1.8,
            marker=analysis.spec.marker,
            markersize=4.8,
            markerfacecolor="white",
            markeredgewidth=1.1,
            label=analysis.spec.label,
        )
        if analysis.spec.slug.startswith("schema_driven"):
            legend_label = analysis.spec.label.replace("SD", "Schema-driven", 1)
        else:
            legend_label = analysis.spec.label.replace("QD", "Query-driven", 1)
        legend_labels.append(legend_label.replace("view", "View"))
        legend_handles.append(line)

    ax.set_xlabel(
        "Budget (number of selected view definitions)",
        fontsize=AXIS_LABEL_FONT_SIZE,
    )
    ax.set_ylabel("Time saved (seconds)", fontsize=AXIS_LABEL_FONT_SIZE)
    ax.set_xlim(-0.25, MAX_VIEW_COUNT + 0.25)
    ax.set_xticks(range(0, MAX_VIEW_COUNT + 1))
    ax.tick_params(axis="both", labelsize=TICK_LABEL_FONT_SIZE)
    ax.grid(True, axis="both", linestyle=":", linewidth=0.75, alpha=0.75)
    ax.legend(
        legend_handles,
        legend_labels,
        loc="center left",
        bbox_to_anchor=LEGEND_ANCHOR,
        ncol=1,
        frameon=False,
        fontsize=LEGEND_FONT_SIZE,
    )
    fig.tight_layout(rect=(0, 0, PLOT_RIGHT_MARGIN, 1))
    fig.savefig(OUTPUT_PLOT, dpi=240, bbox_inches="tight")
    fig.savefig(OUTPUT_PLOT_PDF, bbox_inches="tight")
    return fig


In [ ]:
def write_unified_baseline_csv(baseline_by_query: dict[int, float]) -> None:
    with OUTPUT_BASELINE_CSV.open("w", encoding="utf-8", newline="") as csv_out:
        writer = csv.DictWriter(
            csv_out,
            fieldnames=["query_id", "baseline_seconds"],
        )
        writer.writeheader()
        for qid in WORKLOAD_QIDS:
            writer.writerow(
                {
                    "query_id": qid,
                    "baseline_seconds": round(baseline_by_query[qid], 12),
                }
            )


analyses = [analyze_dataset(spec) for spec in DATASETS]
unified_baseline = load_unified_baseline()
write_unified_baseline_csv(unified_baseline)
write_best_csv(analyses)

summary = build_summary(analyses)
OUTPUT_SUMMARY.write_text(summary, encoding="utf-8")
figure = plot_combined_tradeoff(analyses)

for analysis in analyses:
    previous_runtime = float("inf")
    for budget in range(MAX_VIEW_COUNT + 1):
        best = analysis.best_by_view_count[budget]
        assert best["view_count"] <= budget
        assert best["total_runtime"] <= previous_runtime + 1e-8
        previous_runtime = best["total_runtime"]
    assert abs(analysis.best_by_view_count[0]["saved_time"]) < 1e-8

with OUTPUT_BEST_CSV.open("r", encoding="utf-8", newline="") as csv_in:
    output_rows = list(csv.DictReader(csv_in))
assert len(output_rows) == len(DATASETS) * (MAX_VIEW_COUNT + 1)
assert all(path.is_file() and path.stat().st_size > 0 for path in (
    OUTPUT_PLOT,
    OUTPUT_PLOT_PDF,
    OUTPUT_SUMMARY,
    OUTPUT_BEST_CSV,
    OUTPUT_BASELINE_CSV,
))

print(f"Analyzed {len(analyses)} datasets with {len(output_rows)} budget results.")
print(f"Unified workload baseline: {sum(unified_baseline.values()):.12f}s")
for analysis in analyses:
    best = analysis.best_by_view_count[MAX_VIEW_COUNT]
    print(
        f"{analysis.spec.label}: best@{MAX_VIEW_COUNT}="
        f"{best['total_runtime']:.6f}s ({best['gain_ratio'] * 100:.2f}% gain)"
    )

print("\nGenerated files:")
for output_path in (
    OUTPUT_PLOT,
    OUTPUT_PLOT_PDF,
    OUTPUT_SUMMARY,
    OUTPUT_BEST_CSV,
    OUTPUT_BASELINE_CSV,
):
    print(f"  - {relative_to_analysis(output_path)}")

display(figure)
plt.close(figure)


## 2. Refresh-aware saving/refresh-cost trade-off

In [ ]:
import math

REFRESH_AWARE_INPUT_JSON = (
    DATA_DIR
    / "refresh_aware_tradeoff"
    / "combined_budget_exact_tradeoff_summary.json"
)
REFRESH_AWARE_OUTPUT_DIR = RESULT_DIR / "refresh_aware_tradeoff"
REFRESH_AWARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFRESH_AWARE_OUTPUT_PNG = (
    REFRESH_AWARE_OUTPUT_DIR / "combined_budget_exact_tradeoff_lines.png"
)
REFRESH_AWARE_OUTPUT_PDF = (
    REFRESH_AWARE_OUTPUT_DIR / "combined_budget_exact_tradeoff_lines.pdf"
)
REFRESH_AWARE_OUTPUT_JSON = (
    REFRESH_AWARE_OUTPUT_DIR / "combined_budget_exact_tradeoff_summary.json"
)
REFRESH_AWARE_OUTPUT_MD = (
    REFRESH_AWARE_OUTPUT_DIR / "combined_budget_exact_tradeoff_summary.md"
)

REFRESH_DATASET_STYLES = {
    "schema": {"label": "Schema", "linestyle": "-", "alpha": 0.95},
    "query": {"label": "Query", "linestyle": "--", "alpha": 0.90},
}
REFRESH_METRIC_STYLES = {
    "query_saved_seconds": {
        "label": "Query time saved", "color": "tab:blue",
        "marker": "o", "linewidth": 2.35,
    },
    "mv_refresh_insert_seconds": {
        "label": "Insert refresh cost", "color": "tab:orange",
        "marker": "^", "linewidth": 2.0,
    },
    "mv_refresh_delete_seconds": {
        "label": "Delete refresh cost", "color": "tab:green",
        "marker": "v", "linewidth": 2.0,
    },
    "mv_refresh_seconds": {
        "label": "Total refresh cost", "color": "tab:red",
        "marker": "s", "linewidth": 2.35,
    },
}
REFRESH_EXPECTED_DATASETS = ("schema", "query")
REFRESH_EXPECTED_BUDGETS = list(range(9))


def refresh_budget_rows(dataset_payload: dict[str, Any]) -> list[dict[str, Any]]:
    return sorted(dataset_payload["budgets"], key=lambda row: row["mv_budget"])


def validate_refresh_tradeoff_payload(payload: dict[str, Any]) -> None:
    datasets = payload.get("datasets")
    if not isinstance(datasets, dict):
        raise ValueError("Refresh-aware summary must contain a datasets object.")
    if set(datasets) != set(REFRESH_EXPECTED_DATASETS):
        raise ValueError("Refresh-aware summary must contain schema and query datasets.")

    for dataset_name in REFRESH_EXPECTED_DATASETS:
        rows = refresh_budget_rows(datasets[dataset_name])
        budgets = [row["mv_budget"] for row in rows]
        if budgets != REFRESH_EXPECTED_BUDGETS:
            raise ValueError(
                f"{dataset_name} budgets must be the integers from 0 through 8."
            )

        for metric_name in REFRESH_METRIC_STYLES:
            values = [float(row[metric_name]) for row in rows]
            if any(not math.isfinite(value) or value < 0 for value in values):
                raise ValueError(
                    f"{dataset_name}.{metric_name} contains an invalid value."
                )

        for row in rows:
            budget = int(row["mv_budget"])
            refresh_total = (
                float(row["mv_refresh_insert_seconds"])
                + float(row["mv_refresh_delete_seconds"])
            )
            if not math.isclose(
                float(row["mv_refresh_seconds"]),
                refresh_total,
                rel_tol=1e-9,
                abs_tol=1e-12,
            ):
                raise ValueError(
                    f"Refresh total mismatch for {dataset_name} at budget {budget}."
                )

            selected_views = row["mv_set"]
            if budget == 0 and selected_views != "baseline":
                raise ValueError(f"Budget 0 must use baseline for {dataset_name}.")
            if budget > 0 and (
                not isinstance(selected_views, list)
                or len(selected_views) != budget
            ):
                raise ValueError(
                    f"Selected-view count mismatch for {dataset_name} "
                    f"at budget {budget}."
                )

            ratio = row["tradeoff_ratio_seconds"]
            saved = float(row["query_saved_seconds"])
            if refresh_total == 0 and ratio is not None:
                raise ValueError(
                    f"Zero refresh cost requires a null ratio for {dataset_name}."
                )
            if refresh_total > 0 and not math.isclose(
                float(ratio),
                saved / refresh_total,
                rel_tol=1e-9,
                abs_tol=1e-12,
            ):
                raise ValueError(
                    f"Trade-off ratio mismatch for {dataset_name} at budget {budget}."
                )


def load_refresh_tradeoff_payload() -> dict[str, Any]:
    validate_input_paths((REFRESH_AWARE_INPUT_JSON,))
    payload = json.loads(REFRESH_AWARE_INPUT_JSON.read_text(encoding="utf-8"))
    validate_refresh_tradeoff_payload(payload)
    return payload


In [ ]:
def compact_refresh_budget_rows(
    dataset_payload: dict[str, Any],
) -> list[dict[str, Any]]:
    fields = (
        "mv_budget",
        "query_saved_seconds",
        "mv_refresh_insert_seconds",
        "mv_refresh_delete_seconds",
        "mv_refresh_seconds",
        "tradeoff_ratio_seconds",
        "mv_set",
    )
    return [
        {field: row[field] for field in fields}
        for row in refresh_budget_rows(dataset_payload)
    ]


def build_refresh_summary_payload(
    payload: dict[str, Any],
) -> dict[str, Any]:
    return {
        "source_summary_json": relative_to_analysis(REFRESH_AWARE_INPUT_JSON),
        "output_png": relative_to_analysis(REFRESH_AWARE_OUTPUT_PNG),
        "output_pdf": relative_to_analysis(REFRESH_AWARE_OUTPUT_PDF),
        "datasets": {
            name: {
                "best_tradeoff": payload["datasets"][name].get("best_tradeoff"),
                "budgets": compact_refresh_budget_rows(
                    payload["datasets"][name]
                ),
            }
            for name in REFRESH_EXPECTED_DATASETS
        },
    }


def build_refresh_markdown(summary_payload: dict[str, Any]) -> str:
    lines = [
        "# Combined Budget Exact Trade-off",
        "",
        f"- Source: {summary_payload['source_summary_json']}",
        f"- Plot: {REFRESH_AWARE_OUTPUT_PNG.name}",
        f"- PDF: {REFRESH_AWARE_OUTPUT_PDF.name}",
        "- Solid lines are schema results; dashed lines are query results.",
        (
            "- Curves show query-time saving, insert refresh cost, "
            "delete refresh cost, and total refresh cost."
        ),
        "",
        "| dataset | best b | saving / refresh cost | saving (s) | refresh cost (s) |",
        "|---|---:|---:|---:|---:|",
    ]

    for dataset_name in REFRESH_EXPECTED_DATASETS:
        best = summary_payload["datasets"][dataset_name]["best_tradeoff"]
        ratio = best["tradeoff_ratio_seconds"]
        ratio_text = "N/A" if ratio is None else f"{ratio:.6f}"
        lines.append(
            f"| {dataset_name} | "
            f"{best['mv_budget']} | "
            f"{ratio_text} | "
            f"{best['query_saved_seconds']:.6f} | "
            f"{best['mv_refresh_seconds']:.6f} |"
        )

    return "\n".join(lines) + "\n"


def plot_refresh_aware_tradeoff(payload: dict[str, Any]):
    fig, ax = plt.subplots(figsize=(12.5, 4.3))

    for dataset_name in REFRESH_EXPECTED_DATASETS:
        dataset_style = REFRESH_DATASET_STYLES[dataset_name]
        rows = refresh_budget_rows(payload["datasets"][dataset_name])
        budgets = [row["mv_budget"] for row in rows]

        for metric_name, metric_style in REFRESH_METRIC_STYLES.items():
            values = [row[metric_name] for row in rows]
            ax.plot(
                budgets,
                values,
                color=metric_style["color"],
                linestyle=dataset_style["linestyle"],
                marker=metric_style["marker"],
                linewidth=metric_style["linewidth"],
                alpha=dataset_style["alpha"],
                label=f"{dataset_style['label']} - {metric_style['label']}",
            )

    ax.set_xlabel(
        "Budget b (number of selected view definitions)",
        fontsize=AXIS_LABEL_FONT_SIZE,
    )
    ax.set_ylabel("Saving / refresh cost (s)", fontsize=AXIS_LABEL_FONT_SIZE)
    ax.set_xticks(REFRESH_EXPECTED_BUDGETS)
    ax.set_xlim(-0.15, 8.15)
    ax.tick_params(axis="both", labelsize=TICK_LABEL_FONT_SIZE)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.85)
    ax.legend(loc="upper left", ncol=2, fontsize=11, frameon=False)
    fig.tight_layout()
    fig.savefig(REFRESH_AWARE_OUTPUT_PNG, dpi=220, bbox_inches="tight")
    fig.savefig(REFRESH_AWARE_OUTPUT_PDF, bbox_inches="tight")
    return fig


refresh_tradeoff_payload = load_refresh_tradeoff_payload()
refresh_tradeoff_figure = plot_refresh_aware_tradeoff(refresh_tradeoff_payload)
refresh_summary_payload = build_refresh_summary_payload(refresh_tradeoff_payload)

REFRESH_AWARE_OUTPUT_JSON.write_text(
    json.dumps(refresh_summary_payload, indent=2),
    encoding="utf-8",
)
REFRESH_AWARE_OUTPUT_MD.write_text(
    build_refresh_markdown(refresh_summary_payload),
    encoding="utf-8",
)

refresh_outputs = (
    REFRESH_AWARE_OUTPUT_PNG,
    REFRESH_AWARE_OUTPUT_PDF,
    REFRESH_AWARE_OUTPUT_JSON,
    REFRESH_AWARE_OUTPUT_MD,
)
assert all(path.is_file() and path.stat().st_size > 0 for path in refresh_outputs)

print(
    "Loaded refresh-aware input: "
    f"{relative_to_analysis(REFRESH_AWARE_INPUT_JSON)}"
)
for dataset_name in REFRESH_EXPECTED_DATASETS:
    best = refresh_tradeoff_payload["datasets"][dataset_name]["best_tradeoff"]
    print(
        f"{dataset_name}: best b={best['mv_budget']}, "
        f"saving={best['query_saved_seconds']:.6f}s, "
        f"refresh cost={best['mv_refresh_seconds']:.6f}s, "
        f"ratio={best['tradeoff_ratio_seconds']:.6f}"
    )

print("\nGenerated refresh-aware files:")
for output_path in refresh_outputs:
    print(f"  - {relative_to_analysis(output_path)}")

display(refresh_tradeoff_figure)
plt.close(refresh_tradeoff_figure)
